# 1장 01. 데이터 파일과 컬럼 확인

이 notebook은 평가 데이터 파일을 열고, 모델 평가에 필요한 필수 컬럼이 있는지 확인합니다. 복잡한 품질 리포트는 만들지 않고, 파일 출처, 행 수, 컬럼 목록만 먼저 봅니다.


## 기본 개념: 데이터 파일과 schema

데이터 파일 확인은 MLOps 파이프라인의 가장 앞쪽에 있는 입력 계약 확인입니다. 모델 학습, 평가, 배포 자동화가 아무리 잘 되어 있어도, 다른 파일을 읽거나 컬럼 이름이 바뀌면 뒤 단계의 metric과 로그는 모두 잘못된 근거가 됩니다.

Schema는 데이터가 어떤 컬럼과 타입을 가져야 하는지 정한 약속입니다. 운영에서는 schema가 문서에만 있으면 부족하고, 데이터 로딩 코드, 검증 규칙, 모델 입력 feature 목록, serving request schema가 같은 기준을 바라봐야 합니다.

이 노트북에서는 분석을 깊게 하지 않습니다. MLOps 관점에서 “현재 평가 데이터가 뒤 단계로 넘어갈 수 있는 최소 구조를 갖췄는가”를 확인합니다. 그래서 행 수, 컬럼 수, 필수 컬럼, label 값만 먼저 봅니다.

| 개념 | MLOps에서의 의미 | 이 노트북에서 확인하는 값 |
| --- | --- | --- |
| 데이터 파일 | pipeline의 입력 snapshot | `source`, `shape` |
| Schema | downstream step이 기대하는 컬럼 계약 | 필수 컬럼 존재 여부 |
| Feature | 모델 입력으로 쓰는 값 | `heart_rate`, `oxygen_saturation` 등 |
| Label | metric 계산에 쓰는 정답 기준 | `high_risk`, `low_risk` |

이 단계가 통과되어야 결측 확인, metric 계산, MLflow 기록, serving 후보 비교가 같은 데이터 기준에서 이어집니다. 여기서 틀리면 뒤에서 좋은 모델을 고르는 문제가 아니라, 잘못된 입력을 기준으로 전체 MLOps 흐름이 움직이는 문제가 됩니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


## 1. 평가 데이터 열기

먼저 `vital_signs_evaluation_baseline.csv`를 엽니다. JupyterLite에서 파일이 없으면 작은 sample을 사용합니다.


In [ ]:
dataframe, source = aiq.load_csv_or_sample(
    "data/vital_signs_evaluation_baseline.csv",
    aiq.sample_vital_signs(),
    nrows=200,
)

print("source:", source)
print("rows:", len(dataframe))
print("columns:", len(dataframe.columns))


## 2. 앞부분 보기

`head()`는 전체 분석이 아니라 데이터 모양 확인입니다. 한 row가 어떤 값을 갖는지 먼저 봅니다.


In [ ]:
dataframe.head(5)


## 3. 필수 컬럼 확인

모델 평가에 필요한 컬럼이 있는지 확인합니다. 여기서는 컬럼 이름 존재 여부만 봅니다.


In [ ]:
required = pd.DataFrame({"column": aiq.REQUIRED_COLUMNS})
required["exists"] = required["column"].isin(dataframe.columns)
required
